# Official CODI KV target-utility screen on Kaggle

## Goal

Determine which teacher KV target families produce a locally helpful optimization update before fitting another spectral decomposition or training another student. The primary comparison is correctly paired teacher targets versus both no KV target and shuffled teacher targets under equal parameter-update norm.

This notebook is designed for **Kaggle Save Version → Save & Run All**. Enable **Internet** and a **T4 GPU**. The browser does not need to remain open after the committed Kaggle run starts.

The default run performs the official CODI GSM8K reproduction gate when necessary, an 8-example smoke test, and the preregistered kind-level key-versus-value screen. Position and layer-band refinements are opt-in and should be enabled only after a helpful parent family is found.

## 1. Configure the run

Push the implementation first. Run setup once, copy the printed immutable commit into `RUN_COMMIT`, restart the session, and use that pinned commit for the saved run.

`RESUME_INPUT` may point to an attached dataset containing a previous partial export. `REPRODUCTION_SUMMARY_INPUT` may point to a previously passed official-CODI full-GSM8K `summary.json`; otherwise the notebook creates it.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit printed by setup.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""  # Optional exact summary.json path.
RESUME_INPUT = ""  # Optional attached previous output/dataset root.
RUN_REPRODUCTION_GATE_IF_MISSING = True

RUN_SMOKE = True
RUN_KIND_SCREEN = True
RUN_POSITION_SCREEN = False
RUN_LAYER_BAND_SCREEN = False

# Position screening defaults to all kind-level families classified as helpful.
# Set explicitly, for example ["key"], to override automatic selection.
POSITION_KINDS = []

# Layer-band screening is deliberately explicit because it is the third hierarchy level.
LAYER_BAND_KIND = "key"
LAYER_BAND_POSITIONS = [4]

EXAMPLES_PER_SPLIT = 128
SMOKE_EXAMPLES_PER_SPLIT = 8
BATCH_SIZE = 4
DISCOVERY_SEED = 3
KV_WEIGHT = 1.0
RELATIVE_UPDATE_NORM = 1e-4
METRIC = "l1"
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-kv-target-utility"

## 2. Set up the pinned repository

The official-CODI dependency file matches the author-checkpoint evaluation environment while preserving Kaggle's CUDA-compatible PyTorch build.

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL SAVE VERSION RUN:", commit)

## 3. Verify the GPU and implementation

The current Kaggle PyTorch CUDA build requires a T4-class or newer GPU for this environment. A P100 may fail because its compute capability is no longer included in the runtime build.

In [ ]:
import torch
import transformers

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", gpu_name, "capability:", capability)
assert capability >= (7, 0), (
    f"GPU compute capability {capability} is unsupported by this PyTorch build. "
    "Select a T4 accelerator and restart the Kaggle session."
)

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_official_codi_kv.py",
        "tests/test_kv_target_utility.py",
        "tests/test_official_codi_target_utility.py",
        "tests/test_official_codi_kv_target_utility_analysis.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 4. Prepare durable Kaggle paths and logging

Every batch is written atomically. Rerunning the same experiment directory verifies and skips completed batches.

In [ ]:
WORK_OUTPUT_ROOT = repo / "outputs" / "official_codi_kv_target_utility"
WORK_REPORT_ROOT = repo / "reports" / "official_codi_kv_target_utility"
WORK_LOG_ROOT = repo / "logs" / "official_codi_kv_target_utility"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (WORK_OUTPUT_ROOT, WORK_REPORT_ROOT, WORK_LOG_ROOT, VALIDATION_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(command, log_name):
    log_path = WORK_LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent session log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(
            f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} "
            f"{' '.join(map(str, command))} ===\n"
        )
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
        log.flush()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {log_path}")
    return log_path

## 5. Optional resume from an attached Kaggle dataset

Attach the prior saved notebook output or exported dataset, then set `RESUME_INPUT`. The whole target-utility tree is restored so manifests and completed batch files remain together.

In [ ]:
if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(resume_root.rglob("official_codi_kv_target_utility/kind_seed3/run_manifest.json"))
    assert manifests, "No kind_seed3 target-utility manifest was found in RESUME_INPUT"
    request_hashes = {
        json.loads(path.read_text()).get("request_sha256") for path in manifests
    }
    assert len(request_hashes) == 1, (
        "RESUME_INPUT contains incompatible duplicate runs: "
        f"{[(str(path), json.loads(path.read_text()).get('request_sha256')) for path in manifests]}"
    )
    # A saved notebook output can contain both the work tree and its export copy.
    # They are identical by request hash, so choose the most compact path deterministically.
    source_root = sorted(
        {path.parents[1] for path in manifests},
        key=lambda path: (len(path.parts), path.as_posix()),
    )[0]
    shutil.copytree(source_root, WORK_OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored target-utility outputs from:", source_root)
else:
    print("Starting without a previous target-utility output")

## 6. Locate or create the official-CODI reproduction gate

The utility runner refuses to proceed unless the author-released CODI checkpoint has passed the complete 1,319-example GSM8K gate. If no passed summary is attached, this cell creates one using the official evaluator.

In [ ]:
EXPECTED_CHECKPOINT_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"

def is_passed_reproduction_summary(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    return (
        payload.get("accuracy_gate", {}).get("status") == "passed"
        and payload.get("evaluated_counts", {}).get("gsm8k") == 1319
        and payload.get("checkpoint_revision") == EXPECTED_CHECKPOINT_REVISION
    )

if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT)
    assert is_passed_reproduction_summary(REPRODUCTION_SUMMARY), (
        f"Not a passed official-CODI full-GSM8K summary: {REPRODUCTION_SUMMARY}"
    )
else:
    attached = [
        path for path in pathlib.Path("/kaggle/input").rglob("summary.json")
        if is_passed_reproduction_summary(path)
    ]
    local = [
        path for path in VALIDATION_ROOT.rglob("summary.json")
        if is_passed_reproduction_summary(path)
    ]
    candidates = attached + local
    if candidates:
        REPRODUCTION_SUMMARY = sorted(candidates, key=lambda path: path.as_posix())[0]
    else:
        assert RUN_REPRODUCTION_GATE_IF_MISSING, (
            "No passed official-CODI reproduction summary was found. Attach one or "
            "set RUN_REPRODUCTION_GATE_IF_MISSING=True."
        )
        run_persisted(
            [
                sys.executable, "-u", "-m", "src.eval.official_codi",
                "--config", "configs/official_codi_gpt2.yaml",
                "--datasets", "gsm8k",
                "--limit", "0",
                "--device", "cuda",
                "--output-dir", str(VALIDATION_ROOT),
            ],
            "official_codi_gsm8k_gate.log",
        )
        candidates = [
            path for path in VALIDATION_ROOT.rglob("summary.json")
            if is_passed_reproduction_summary(path)
        ]
        assert len(candidates) == 1, f"Expected one new passed reproduction summary, found {candidates}"
        REPRODUCTION_SUMMARY = candidates[0]

print("Reproduction summary:", REPRODUCTION_SUMMARY)
print(json.dumps(json.loads(REPRODUCTION_SUMMARY.read_text()), indent=2))

## 7. Build one target-utility command

The helper keeps the scientific contract identical across the smoke, kind, position, and layer-band screens.

In [ ]:
def utility_command(output_dir, *, examples, granularity, kinds, positions):
    return [
        sys.executable, "-u", "scripts/run_official_codi_kv_target_utility.py",
        "--config", "configs/official_codi_gpt2.yaml",
        "--reproduction-summary", str(REPRODUCTION_SUMMARY),
        "--output-dir", str(output_dir),
        "--examples-per-split", str(examples),
        "--batch-size", str(BATCH_SIZE),
        "--granularity", granularity,
        "--kinds", ",".join(kinds),
        "--positions", ",".join(map(str, positions)),
        "--metric", METRIC,
        "--kv-weight", str(KV_WEIGHT),
        "--relative-update-norm", str(RELATIVE_UPDATE_NORM),
        "--precision", PRECISION,
        "--device", "cuda",
        "--seed", str(DISCOVERY_SEED),
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
        "--bootstrap-seed", str(BOOTSTRAP_SEED),
    ]

ALL_POSITIONS = [0, 1, 2, 3, 4, 5]

## 8. Run the smoke test

This verifies the official checkpoint, six-step teacher/student KV alignment, stateless virtual update, shuffled-pairing control, held-out answer loss, and report generation before the full screen.

In [ ]:
from IPython.display import Markdown, display

SMOKE_ROOT = WORK_OUTPUT_ROOT / "smoke_kind_seed3"
if RUN_SMOKE:
    run_persisted(
        utility_command(
            SMOKE_ROOT,
            examples=SMOKE_EXAMPLES_PER_SPLIT,
            granularity="kind",
            kinds=["key", "value"],
            positions=ALL_POSITIONS,
        ),
        "smoke_kind_seed3.log",
    )
    smoke_manifest = json.loads((SMOKE_ROOT / "run_manifest.json").read_text())
    assert smoke_manifest["state"] == "complete"
    assert len(smoke_manifest["completed_batches"]) == 2
    display(Markdown((SMOKE_ROOT / "report.md").read_text()))
else:
    print("Smoke test skipped")

## 9. Run the preregistered kind-level screen

This is the required first hierarchy level. Keys and values are evaluated on the same discovery/validation split with matched update magnitude and shuffled targets.

In [ ]:
KIND_ROOT = WORK_OUTPUT_ROOT / "kind_seed3"
if RUN_KIND_SCREEN:
    run_persisted(
        utility_command(
            KIND_ROOT,
            examples=EXAMPLES_PER_SPLIT,
            granularity="kind",
            kinds=["key", "value"],
            positions=ALL_POSITIONS,
        ),
        "kind_seed3.log",
    )
    kind_manifest = json.loads((KIND_ROOT / "run_manifest.json").read_text())
    kind_report = json.loads((KIND_ROOT / "summary.json").read_text())
    assert kind_manifest["state"] == "complete"
    assert len(kind_manifest["completed_batches"]) == EXAMPLES_PER_SPLIT // BATCH_SIZE
    display(Markdown((KIND_ROOT / "report.md").read_text()))
else:
    assert (KIND_ROOT / "summary.json").is_file(), (
        "RUN_KIND_SCREEN=False requires an existing completed kind_seed3 summary"
    )
    kind_report = json.loads((KIND_ROOT / "summary.json").read_text())

helpful_kind_groups = [
    name for name, classification in kind_report["classifications"].items()
    if classification == "helpful_target_family"
]
helpful_kinds = [name.removesuffix("_all") for name in helpful_kind_groups]
print("Kind-level screen status:", kind_report["screen_status"])
print("Helpful kinds eligible for position refinement:", helpful_kinds)

## 10. Optional position refinement

Enable only when the kind-level screen finds a helpful parent family. With `POSITION_KINDS = []`, the cell automatically uses those helpful kinds. It refuses to search positions when no parent kind passed.

In [ ]:
POSITION_ROOT = WORK_OUTPUT_ROOT / "position_seed3"
selected_position_kinds = POSITION_KINDS or helpful_kinds
if RUN_POSITION_SCREEN:
    assert selected_position_kinds, (
        "No helpful kind-level family is available for position refinement. "
        "Do not bypass the hierarchy without preregistering a separate diagnostic run."
    )
    assert set(selected_position_kinds).issubset(set(helpful_kinds)), (
        f"POSITION_KINDS must be a subset of helpful kinds {helpful_kinds}"
    )
    run_persisted(
        utility_command(
            POSITION_ROOT,
            examples=EXAMPLES_PER_SPLIT,
            granularity="position",
            kinds=selected_position_kinds,
            positions=ALL_POSITIONS,
        ),
        "position_seed3.log",
    )
    position_report = json.loads((POSITION_ROOT / "summary.json").read_text())
    display(Markdown((POSITION_ROOT / "report.md").read_text()))
else:
    position_report = None
    print("Position screen skipped. Enable it only after reviewing the kind-level report.")

## 11. Optional layer-band refinement

This third level evaluates early layers 0–3, middle layers 4–7, and late layers 8–11 for one explicitly selected kind and one or more positions. It requires those exact position families to have passed the position screen.

In [ ]:
BAND_ROOT = WORK_OUTPUT_ROOT / f"{LAYER_BAND_KIND}_bands_seed3"
if RUN_LAYER_BAND_SCREEN:
    assert position_report is not None, "Run the position screen before layer-band refinement"
    required_groups = {f"{LAYER_BAND_KIND}_p{position}" for position in LAYER_BAND_POSITIONS}
    passed_groups = {
        name for name, classification in position_report["classifications"].items()
        if classification == "helpful_target_family"
    }
    assert required_groups.issubset(passed_groups), (
        f"Layer-band parents {sorted(required_groups)} did not all pass. "
        f"Helpful position groups are {sorted(passed_groups)}"
    )
    run_persisted(
        utility_command(
            BAND_ROOT,
            examples=EXAMPLES_PER_SPLIT,
            granularity="layer_band",
            kinds=[LAYER_BAND_KIND],
            positions=LAYER_BAND_POSITIONS,
        ),
        f"{LAYER_BAND_KIND}_bands_seed3.log",
    )
    band_report = json.loads((BAND_ROOT / "summary.json").read_text())
    display(Markdown((BAND_ROOT / "report.md").read_text()))
else:
    band_report = None
    print("Layer-band screen skipped.")

## 12. Checks and decision boundary

The notebook reports a short-horizon optimization diagnostic. A helpful family may advance to position refinement and then answer-conditioned spectral analysis. If neither kind is helpful, stop this target definition rather than fitting another denoiser.

In [ ]:
print("\nFINAL COMPLETED SCREENS")
for name, root in [
    ("smoke", SMOKE_ROOT),
    ("kind", KIND_ROOT),
    ("position", POSITION_ROOT),
    ("layer_band", BAND_ROOT),
]:
    summary_path = root / "summary.json"
    if summary_path.is_file():
        payload = json.loads(summary_path.read_text())
        print(f"{name:12s}", payload["screen_status"], payload["classifications"])

if not helpful_kinds:
    print(
        "\nDECISION: Neither key nor value passed the kind-level utility gate. "
        "Stop this KV target definition and do not launch spectral distillation training."
    )
else:
    print(
        "\nDECISION: Only these kinds may proceed to position refinement:",
        helpful_kinds,
    )

## 13. Build a durable Kaggle export

The export includes manifests, completed batch records, reports, logs, the official reproduction summary, the exact commit, and SHA-256 checksums. It excludes Hugging Face caches and model downloads.

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_kv_target_utility_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(WORK_OUTPUT_ROOT, export_repo / "outputs" / "official_codi_kv_target_utility")
shutil.copytree(WORK_LOG_ROOT, export_repo / "logs" / "official_codi_kv_target_utility")
if WORK_REPORT_ROOT.exists():
    shutil.copytree(WORK_REPORT_ROOT, export_repo / "reports" / "official_codi_kv_target_utility")
validation_export = export_repo / "outputs" / "official_codi_gpt2_reproduction"
validation_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPRODUCTION_SUMMARY, validation_export / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text(
    "Attach this dataset and set RESUME_INPUT to its root to continue optional hierarchy levels.\n"
)

files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
checksum_lines = []
for path in files:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksum_lines.append(f"{digest}  {path.relative_to(EXPORT_ROOT).as_posix()}")
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksum_lines) + "\n")

all_files = [path for path in EXPORT_ROOT.rglob("*") if path.is_file()]
print("Export root:", EXPORT_ROOT)
print("Files:", len(all_files))
print("Size:", sum(path.stat().st_size for path in all_files) / 2**20, "MiB")
print("Use Save Version with outputs enabled. The browser may be closed after the committed run starts.")

## 14. Optional direct Kaggle Dataset upload

Saving the notebook version with outputs is sufficient. Enable direct upload only after the expected screen is complete and the configured Kaggle account owns the dataset handle.

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        KAGGLE_DATASET_HANDLE,
        str(EXPORT_ROOT),
        version_notes=f"Official CODI KV target utility at {commit}",
    )
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct dataset upload skipped. Save this notebook version with outputs enabled.")

## Next steps

1. Read `kind_seed3/report.md`.
2. If a kind is `helpful_target_family`, enable the position screen for that kind and rerun from the configuration cell.
3. Refine only helpful positions into layer bands.
4. Apply answer-conditioned spectral decomposition only inside a family that survives all enabled hierarchy levels.
5. Require an exact held-out causal intervention before committing to expensive distillation training.